[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [scikit-learn, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)

# Imbalanced Classes &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import warnings

import numpy as np
import pandas as pd
import sklearn
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
                             confusion_matrix, precision_score, recall_score)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def station_network(n=1200, seed=2025):
    """A made-up network of weather stations in Norway, one row each, with three things to predict."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    terrain = np.where(coast_km < 3, "coastal",
              np.where(elevation > 700, "mountain",
              np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), "fjord", "inland")))
    owner = rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2])
    installed = rng.integers(1950, 2025, n)
    shift = pd.Series(terrain).map({"coastal": 0.8, "fjord": 0.3, "inland": 0.0, "mountain": -0.5}).to_numpy()
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    june_min = 8.5 - 0.35 * (latitude - 58) - 0.0075 * elevation - 0.02 * coast_km + rng.normal(0, 1.5, n)
    table = pd.DataFrame({
        "station": [f"ST{i:04d}" for i in range(1, n + 1)],
        "latitude": latitude.round(2), "longitude": longitude.round(2),
        "elevation_m": elevation.round(0).astype(int), "coast_km": coast_km.round(1),
        "terrain": terrain, "owner": owner, "installed": installed,
        "jan_mean": jan_mean.round(1),
        "snow_on_may_1": may_mean < 2.5,
        "june_frost": june_min < -0.8,
    })
    return table


network = station_network()
FEATURES = ["latitude", "longitude", "elevation_m", "coast_km", "installed"]

print("scikit-learn", sklearn.__version__, "|", len(network), "stations")

scikit-learn 1.6.1 | 1200 stations


**1.** Fit a `DummyClassifier(strategy="stratified", random_state=0)`, which guesses frost at random in proportion to its share, and print its accuracy and recall on the test set.

In [2]:
X = network[FEATURES]
y = network["june_frost"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)

guesser = DummyClassifier(strategy="stratified", random_state=0).fit(X_train, y_train)
guesses = guesser.predict(X_test)

print("accuracy:", round(accuracy_score(y_test, guesses), 3))
print("recall:  ", round(recall_score(y_test, guesses), 3))

accuracy: 0.903
recall:   0.133


The first three lines rebuild the split as the worked examples did, since this notebook runs on its own. Guessing frost at random about one time in twenty found 2 of the 15 frosts, a recall of 0.133, and its accuracy, 0.903, is below the baseline's.

**2.** Print the unweighted model's precision and recall at a threshold of 0.2, using `predict_proba`, and compare them with the weighted model's at 0.5.

In [3]:
X = network[FEATURES]
y = network["june_frost"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)

model = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_train, y_train)
weighted = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced")).fit(X_train, y_train)
at_two_tenths = model.predict_proba(X_test)[:, 1] > 0.2
weighted_says = weighted.predict(X_test)

print("unweighted at 0.2: precision", round(precision_score(y_test, at_two_tenths), 3),
      "| recall", round(recall_score(y_test, at_two_tenths), 3))
print("weighted at 0.5:   precision", round(precision_score(y_test, weighted_says), 3),
      "| recall", round(recall_score(y_test, weighted_says), 3))

unweighted at 0.2: precision 0.579 | recall 0.733
weighted at 0.5:   precision 0.31 | recall 0.867


Lowering the threshold is the other way to find more frosts. At 0.2 the unweighted model has a precision of 0.579 and a recall of 0.733. Against the weighted model's 0.31 and 0.867 at 0.5, its warnings are right more often and it finds fewer frosts.

**3.** Cross-validate the unweighted model with `StratifiedKFold(5, shuffle=True, random_state=0)` and `scoring="balanced_accuracy"`, and print the mean.

In [4]:
X = network[FEATURES]
y = network["june_frost"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)

model = make_pipeline(StandardScaler(), LogisticRegression())
scores = cross_val_score(model, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=0),
                         scoring="balanced_accuracy")

print(scores.round(3), "| mean:", round(scores.mean(), 3))

[0.719 0.706 0.664 0.664 0.787] | mean: 0.708


Every fold is scored on balanced accuracy, so the frost stations weigh as much as the clear ones.

**4.** Make a table with one row for each of the three models, the baseline, the model and the weighted model, and columns for accuracy, balanced accuracy, recall and precision on the test set, with `zero_division=0` for the baseline's precision.

In [5]:
X = network[FEATURES]
y = network["june_frost"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)

models = {
    "baseline": DummyClassifier(strategy="most_frequent"),
    "model": make_pipeline(StandardScaler(), LogisticRegression()),
    "weighted": make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced")),
}
rows = {}
for name, estimator in models.items():
    says = estimator.fit(X_train, y_train).predict(X_test)
    rows[name] = {"accuracy": accuracy_score(y_test, says), "balanced": balanced_accuracy_score(y_test, says),
                  "recall": recall_score(y_test, says), "precision": precision_score(y_test, says, zero_division=0)}

pd.DataFrame(rows).T.round(3)

,accuracy,balanced,recall,precision
baseline,0.950,0.500,0.000,0.000
model,0.973,0.765,0.533,0.889
weighted,0.897,0.882,0.867,0.310


Accuracy puts the weighted model last and recall puts it first, which is the notebook's point in one table.

---

&#8592; **Back to:** [Imbalanced Classes](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/09-imbalanced-classes.ipynb)  &nbsp;&middot;&nbsp;  [scikit-learn, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)
